# İlk Random Forest: eğitimde öğren, doğrulamada incele

Bu defter mevcut `content-split-v1` ayrımındaki **604 eğitim** ve **197 doğrulama** kaydıyla ilk deneyin adımlarını gösterir. **199 test kaydı final değerlendirme için ayrılmıştır; defter test özelliklerini açmaz.** Eski endpoint gruplaması bu deneyin ayrımı değildir.

Hücreleri sırayla çalıştırın. Model ve ön işleme bellekte yeniden kurulur; bu defter dosya kaydetmez, kayıtlı modeli veya deney raporunu değiştirmez. Aşağıdaki ölçümler doğrulama sonuçlarıdır; test başarımı olarak yorumlanmaz.

## 1. Proje kökünü bul ve mevcut API'yi kullan

Defter proje kökünden veya `notebooks/` dizininden başlatılabilir. Eğitim betiğinin fonksiyonlarını içe aktarmak, aynı özelliğin iki farklı biçimde işlenmesini önler. `run_experiment` çağrılmayacak: adımları ayrı ayrı göreceğiz ve yeni deney dosyaları yazmayacağız.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts/train_random_forest.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "scripts/train_random_forest.py").is_file():
    raise FileNotFoundError("Defteri proje kökünden veya notebooks/ dizininden başlatın.")

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyClassifier
from train_random_forest import (
    FEATURE_NAMES, RF_PARAMS, read_jsonl, rows_to_xy,
    make_pipeline, binary_metrics,
)

SPLIT_DIR = PROJECT_ROOT / "data/processed/http-mini-splits-v1"
print("Proje:", PROJECT_ROOT)
print("Girdi özellik sayısı:", len(FEATURE_NAMES))


## 2. Şimdilik yalnız eğitim verisini oku

`rows_to_xy` açıkça tanımlanmış 16 temel + 5 örüntü özelliğinden `X` oluşturur; `y`, 0/1 etiketidir. Kaynak dosya adı, kaynak sıra numarası, `sample_id`, `group_id` veya başka metadata `X`'e girmez. Kimlikler yalnız kayıt eşleştirme ve sonraki hata incelemesinde kullanılabilir.

**0 = normal; 1 = anomalous.** Pozitif sınıf, CSIC veri kümesinin anormal etiketidir; tek başına doğrulanmış saldırı kanıtı değildir. Şema, etiket ve sonlu sayısal değer denetimi başarısız olursa işlem durur.

In [ ]:
train_rows = read_jsonl(SPLIT_DIR / "train_features.jsonl")
X_train, y_train = rows_to_xy(train_rows)
assert X_train.shape == (604, 21)
assert list(X_train.columns) == list(FEATURE_NAMES)
assert set(y_train) == {0, 1}

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
display(pd.Series(y_train, name="etiket").value_counts().sort_index().rename("kayıt sayısı").to_frame())
display(X_train.head())


## 3. Ön işleme ve model aynı Pipeline içinde

Üç adım vardır:

1. **Sonlu değer denetimi:** `NaN`, sonsuz veya eksik değer kabul edilmez. Eksik değer doldurma uygulanmaz; böyle bir veri sorunu sessizce gizlenmez. Denetim veri dağılımından bir doldurma değeri öğrenmez.
2. **Sabit sütunları çıkarma:** `VarianceThreshold(0.0)`, hangi sütunların sabit olduğunu yalnız `X_train` ile öğrenir. Doğrulamada aynı sütun maskesi uygulanır; yeniden `fit` yapılmaz.
3. **Random Forest:** kalan sütunlarla ağaçlar öğrenilir.

Karar ağaçları özelliklerde eşik bölmeleri yaptığı için burada standartlaştırma/ölçekleme gerekmiyor. Ayrım yapıldıktan sonra tüm veride ön işleme öğrenmek bilgi sızıntısına yol açabilir. [scikit-learn: veri sızıntısı ve Pipeline](https://scikit-learn.org/stable/common_pitfalls.html), [karar ağaçları](https://scikit-learn.org/stable/modules/tree.html).

İlk deney için önceden belirlenmiş ayarlar: **100 ağaç**, **en fazla 10 derinlik**, **yaprak başına en az 5 eğitim kaydı**, **tohum 42**, **2 paralel işçi**. Ağaç sayısı topluluk tahminini oluşturur; derinlik ve yaprak sınırları küçük örneklemde aşırı karmaşıklığı sınırlamayı amaçlar. Tohum tekrarlanabilirlik, işçi sayısı kaynak kullanımı içindir. Bu değerler doğrulama sonucuna göre aranmış veya en iyi olarak kanıtlanmış değildir. [RandomForestClassifier parametreleri](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html).

In [ ]:
display(pd.Series(RF_PARAMS, name="değer").to_frame())
pipeline = make_pipeline().fit(X_train, y_train)
print("Pipeline yalnız eğitim verisiyle öğrenildi.")


## 4. Eğitimde öğrenilen sütun seçimini gör

21 aday özellikten eğitimde sabit olan 3 sütunun çıkarılmasıyla **18 sütun** kalır. Bu, elle doğrulamaya bakılarak yapılan bir özellik seçimi değildir. Eğitimde bilgi taşımayan sütunları kaldıran ilk ön işleme adımıdır. Bir sütun başka veride değişse bile mevcut seçici onu çıkarmayı sürdürür.

İlk model **16 temel + 5 örüntü** adayının tamamıyla başlar. Yalnız 16 temel özellikle ayrı model kurup karşılaştırma (*ablation*), henüz yapılmamış sonraki bir deneydir; mevcut sonuç örüntü özelliklerinin katkısını tek başına kanıtlamaz.

In [ ]:
selector = pipeline.named_steps["drop_constant"]
retained_features = list(selector.get_feature_names_out())
dropped_features = [name for name in FEATURE_NAMES if name not in retained_features]
constant_values = pd.DataFrame({
    "özellik": dropped_features,
    "eğitimdeki sabit değer": [float(X_train[name].iloc[0]) for name in dropped_features],
})
assert len(retained_features) == 18
print(f"{len(FEATURE_NAMES)} aday → {len(retained_features)} kalan özellik")
display(constant_values)
display(pd.DataFrame({"modele giren özellik": retained_features}))


## 5. Öğrenme bittikten sonra doğrulamayı aç

Doğrulama kümesi ön işlemenin veya ormanın `fit` işlemine katılmaz. `predict`, eğitimde öğrenilen denetim ve sütun seçimini uygular. Etiketler, tahminler oluştuktan sonra karşılaştırma için kullanılır.

`predict_proba` çıktısındaki sınıf 1 değeri burada **anomalous skoru** diye adlandırılır. Kalibrasyon yapılmadığından gerçek trafikte saldırı olasılığı olarak okunmamalıdır. İki sınıflı bu modelin varsayılan tahmininde skor **0,5'ten büyükse 1**, tam **0,5 ise 0 (normal)** seçilir. Eşik ayarı yapılmamıştır.

In [ ]:
validation_rows = read_jsonl(SPLIT_DIR / "validation_features.jsonl")
X_validation, y_validation = rows_to_xy(validation_rows)
assert X_validation.shape == (197, 21)
assert set(row["sample_id"] for row in train_rows).isdisjoint(
    row["sample_id"] for row in validation_rows
)
assert set(row["group_id"] for row in train_rows).isdisjoint(
    row["group_id"] for row in validation_rows
)

predicted = pipeline.predict(X_validation)
class1_column = list(pipeline.classes_).index(1)
anomalous_scores = pipeline.predict_proba(X_validation)[:, class1_column]
assert np.array_equal(predicted, (anomalous_scores > 0.5).astype(int))
rf_metrics = binary_metrics(y_validation, predicted)
print("Doğrulama kayıt sayısı:", len(y_validation))


## 6. Basit çoğunluk sınıfı yaklaşımıyla karşılaştır

`DummyClassifier(strategy="most_frequent")` yalnız eğitim etiketlerinden çoğunluk sınıfını öğrenir ve her kayıt için onu tahmin eder. Böylece ormanın, özellik kullanmayan basit bir başlangıcın üzerine ne kattığını görürüz. Dummy de doğrulamada yeniden öğrenmez.

Pozitif sınıf 1 için **precision**, anormal denen kayıtların ne kadarının anormal etiketli olduğunu; **recall**, anormal etiketlilerin ne kadarının bulunduğunu gösterir. **F1** bu ikisini birlikte özetler. **Yanlış pozitif oranı**, normal etiketli kayıtların ne kadarına yanlış alarm üretildiğidir. Accuracy tek başına bu hataların türünü göstermez.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
dummy_predicted = dummy.predict(X_validation)
dummy_metrics = binary_metrics(y_validation, dummy_predicted)
metric_columns = ["accuracy", "precision", "recall", "f1", "false_positive_rate"]
comparison = pd.DataFrame({
    "Random Forest": {key: rf_metrics[key] for key in metric_columns},
    "Dummy (eğitim çoğunluğu)": {key: dummy_metrics[key] for key in metric_columns},
}).T
print("Dummy'nin öğrendiği çoğunluk sınıfı:", int(dummy.predict(X_train.iloc[:1])[0]))
display(comparison.round(4))


## 7. Karışıklık matrisi ve hata kayıtları

Matrisin **satırları gerçek etiketi**, **sütunları tahmini** gösterir: sol üst TN, sağ üst FP, sol alt FN, sağ alt TP. FP normal etiketli kayda yanlış alarm; FN anormal etiketli kaydı kaçırmadır.

Aşağıdaki hata tablosundaki `sample_id`, hatayı kaynağa geri bağlamak içindir; model girdisine eklenmez. Skorlar yalnız incelemeyi kolaylaştırır. Bu hücre hata örneklerine bakarak modeli veya eşiği yeniden ayarlamaz.

In [ ]:
confusion_df = pd.DataFrame(
    rf_metrics["confusion_matrix"],
    index=["gerçek 0: normal", "gerçek 1: anomalous"],
    columns=["tahmin 0: normal", "tahmin 1: anomalous"],
)
display(confusion_df)

inspection = pd.DataFrame({
    "sample_id": [row["sample_id"] for row in validation_rows],
    "gerçek_etiket": y_validation,
    "tahmin": predicted,
    "anomalous_skoru": anomalous_scores,
})
errors = inspection.loc[inspection["gerçek_etiket"] != inspection["tahmin"]].copy()
errors["hata_türü"] = np.where(errors["gerçek_etiket"] == 0, "FP", "FN")
errors = errors[["sample_id", "gerçek_etiket", "tahmin", "anomalous_skoru", "hata_türü"]]
print("Hata sayısı:", len(errors))
display(errors)


## İlk çalıştırmanın sonucu ve sınırlar

27 Eylül 2026 tarihli `models/rf-baseline-v1/run.json` kaydında Random Forest için **accuracy %74,62**, **precision %78,57**, **recall %67,35**, **F1 0,7253** ve **yanlış pozitif oranı %18,18** elde edildi. Matris **TN=81, FP=18, FN=32, TP=66**: 197 doğrulama kaydının 50'sinde hata var. Yukarıdaki hücreler aynı kod, veri ve paket ortamında bu sonucu yeniden üretir.

Dummy eğitim çoğunluğu olan 1'i sürekli tahmin etti: accuracy **%49,75**, precision **%49,75**, recall **%100**, F1 **0,6644**, yanlış pozitif oranı **%100**. Bu örnek, yüksek recall'ın tek başına iyi bir IDS anlamına gelmediğini gösterir. Orman bu doğrulama kümesinde daha yüksek accuracy ve F1 sağladı; yine de 98 anormal kaydın 32'sini kaçırdı ve 99 normal kaydın 18'inde yanlış alarm verdi.

Bu, CSIC 2010'dan seçilmiş küçük ve sınıf dağılımı yaklaşık dengeli sentetik bir pilot üzerindeki **geçici doğrulama sonucudur**. Gerçek trafik başarımı iddiası değildir. İçerik gruplaması sezgisel kurallara dayanır; endpoint paylaşımı bilinçli olarak mümkündür. Özelliklerin tüm veri üzerinde daha önce incelenmiş olması da deneyin sınırlılığı olarak kaydedilmiştir. `anomalous` etiketi saldırının doğrulandığı anlamına gelmez.

**Henüz yapılmayanlar:** yalnız temel özelliklerle karşılaştırma, hiperparametre araması, eşik ayarı, olasılık kalibrasyonu ve final test değerlendirmesi. Mevcut ayrım korunur. 199 test kaydı, deney tercihleri sabitlenene kadar kapalı kalır. Bu defter kayıtlı model dosyalarını değiştirmez.